# Autoencoder for Anomaly Detection

## Overview
This notebook implements an **Autoencoder neural network** for detecting anomalies (fake news) in text data using **Deep Learning features** (padded token sequences with embedding layer).

### Key Components:
- **Data**: Generate DL features (tokenized & padded sequences) from processed text
- **Embedding**: Learned embeddings that convert tokens to dense vectors
- **Model**: Autoencoder with embedding layer + dense encoder-decoder
- **Training**: Train on real news only (label 0) to establish normal reconstruction patterns
- **Detection**: Use reconstruction error as anomaly score to identify outliers
- **Evaluation**: Compare detected anomalies with ground truth labels

### Advantages:
- Uses pre-processed text data (not raw TF-IDF)
- Embedding layer learns semantic representations
- More suitable for text anomaly detection
- Captures word context and relationships

### Workflow:
1. Load and preprocess text data using dl_text_preprocessing
2. Generate padded sequences
3. Build autoencoder with embedding layer
4. Train on normal samples
5. Calculate reconstruction errors
6. Define threshold and detect anomalies
7. Evaluate and save results

In [13]:
import numpy as np
import os
from pathlib import Path
from joblib import load, dump
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

BASE_DIR = Path(os.getcwd())
while BASE_DIR.name != "nlp-fake-news-detector-transformers" and BASE_DIR.parent != BASE_DIR:
    BASE_DIR = BASE_DIR.parent

FEATURES_DIR = BASE_DIR / "data" / "saved_features"
MODELS_DIR = BASE_DIR / "results" / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

## Generate Deep Learning Features

Load processed text data and generate tokenized + padded sequences using the preprocessing utilities.

In [14]:
import sys
sys.path.insert(0, str(BASE_DIR / "src" / "features"))

from dl_text_preprocessing import (load_clean_data, split_data, build_tokenizer, 
                                    text_to_sequence, apply_padding)
import pandas as pd

# Load and preprocess data
print("Loading and preprocessing text data...")
data_path = BASE_DIR / "data" / "processed" / "processed.csv"
df = load_clean_data(data_path)

# Split data
X_train, X_val, X_test, y_train, y_val, y_test = split_data(df)

# Tokenization (use same defaults as feature_engineering_dl.ipynb)
# Default: max_words=20000, max_len=20
max_words = 20000
max_len = 20
print(f"Building tokenizer (max_words={max_words})...")
tokenizer = build_tokenizer(X_train)  # Uses default max_words=20000

# Convert to sequences
print("Converting text to sequences...")
X_train_seq, X_val_seq, X_test_seq = text_to_sequence(tokenizer, X_train, X_val, X_test)

# Apply padding
print(f"Padding sequences (max_len={max_len})...")
X_train_pad, X_val_pad, X_test_pad = apply_padding(X_train_seq, X_val_seq, X_test_seq)  # Uses default max_len=20

print("\n✓ Data preprocessing completed!")
print(f"X_train_pad shape: {X_train_pad.shape}")
print(f"X_test_pad shape: {X_test_pad.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")
print(f"Vocabulary size: {len(tokenizer.word_index)}")
print(f"Max token index: {max(tokenizer.word_index.values())}")

Loading and preprocessing text data...
Building tokenizer (max_words=20000)...
Converting text to sequences...
Padding sequences (max_len=20)...

✓ Data preprocessing completed!
X_train_pad shape: (963423, 20)
X_test_pad shape: (296438, 20)
y_train shape: (963423,)
y_test shape: (296438,)
Vocabulary size: 295251
Max token index: 295251


## Prepare Data for Autoencoder

Combine training and validation data, convert labels, and prepare for embedding layer input.

In [15]:
# Combine train and validation data for larger training set
X_train_combined = np.vstack([X_train_pad, X_val_pad])
y_train_combined = np.hstack([y_train, y_val])

print("Data preparation for autoencoder:")
print(f"Combined training data shape: {X_train_combined.shape}")
print(f"Test data shape: {X_test_pad.shape}")
print(f"\nClass distribution (combined train):")
print(f"  Real news (0): {(y_train_combined == 0).sum()}")
print(f"  Fake news (1): {(y_train_combined == 1).sum()}")

# Use combined train for autoencoder training
X_train = X_train_combined
y_train = y_train_combined
X_test = X_test_pad

print(f"\nFinal shapes:")
print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"Sequence length: {max_len}")
print(f"Vocabulary size: {max_words}")

Data preparation for autoencoder:
Combined training data shape: (1185752, 20)
Test data shape: (296438, 20)

Class distribution (combined train):
  Real news (0): 599731
  Fake news (1): 586021

Final shapes:
X_train: (1185752, 20)
X_test: (296438, 20)
Sequence length: 20
Vocabulary size: 20000


In [16]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, roc_curve, auc
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


## Build Autoencoder with Embedding Layer

Create an autoencoder that learns embeddings and detects anomalies through reconstruction error.

In [17]:
embedding_dim = 64  # Embedding dimension
encoding_dim = 32   # Bottleneck dimension

print(f"Architecture Configuration:")
print(f"  Sequence length: {max_len}")
print(f"  Vocabulary size: {max_words}")
print(f"  Embedding dimension: {embedding_dim}")
print(f"  Encoding dimension: {encoding_dim}")

# ===== ENCODER =====
encoder_input = layers.Input(shape=(max_len,))

# Embedding layer - converts token indices to dense vectors
embedded = layers.Embedding(input_dim=max_words, output_dim=embedding_dim, 
                           input_length=max_len)(encoder_input)

# Flatten embeddings
flat = layers.Flatten()(embedded)

# Dense encoder
encoded = layers.Dense(128, activation='relu')(flat)
encoded = layers.Dense(64, activation='relu')(encoded)
encoded = layers.Dense(encoding_dim, activation='relu')(encoded)

encoder = Model(encoder_input, encoded, name='encoder')

# ===== DECODER =====
decoder_input = layers.Input(shape=(encoding_dim,))

# Dense decoder
decoded = layers.Dense(64, activation='relu')(decoder_input)
decoded = layers.Dense(128, activation='relu')(decoded)
decoded = layers.Dense(max_len * embedding_dim, activation='relu')(decoded)

# Reshape back to sequence shape
decoded = layers.Reshape((max_len, embedding_dim))(decoded)

# Flatten for final output
decoded = layers.Flatten()(decoded)

decoder = Model(decoder_input, decoded, name='decoder')

# ===== FULL AUTOENCODER =====
autoencoder_input = layers.Input(shape=(max_len,))
encoded_seq = encoder(autoencoder_input)
decoded_seq = decoder(encoded_seq)

autoencoder = Model(autoencoder_input, decoded_seq, name='autoencoder')
autoencoder.compile(optimizer='adam', loss='mse')

print("\n=== Autoencoder Architecture ===")
autoencoder.summary()

Architecture Configuration:
  Sequence length: 20
  Vocabulary size: 20000
  Embedding dimension: 64
  Encoding dimension: 32

=== Autoencoder Architecture ===


Architecture Configuration:
  Sequence length: 20
  Vocabulary size: 20000
  Embedding dimension: 64
  Encoding dimension: 32

=== Autoencoder Architecture ===


Model: "autoencoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_12 (InputLayer)     │ (None, 20)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder (Functional)            │ (None, 32)             │     1,454,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder (Functional)            │ (None, 1280)           │       175,552 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,629,856 (6.22 MB)

 Trainable params: 1,629,856 (6.22 MB)

 Non-trainable params: 0 (0.00 B)

## Train Autoencoder on Normal Data

Train the autoencoder using only real news samples to learn normal reconstruction patterns.

In [ ]:
# Get normal samples (token sequences, not embedded)
normal_idx = np.where(y_train == 0)[0]
X_train_normal = X_train[normal_idx]
print(f"Training samples (real news only): {X_train_normal.shape[0]}")

# Build end-to-end autoencoder with embedding layer
# Input: token sequences [batch, max_len]
# Output: reconstructed embeddings space
print("\nBuilding end-to-end autoencoder with embedding layer...")

# ===== FULL AUTOENCODER =====
autoencoder_input = layers.Input(shape=(max_len,))

# Encoder path
embedded = layers.Embedding(input_dim=max_words, output_dim=embedding_dim)(autoencoder_input)
flat = layers.Flatten()(embedded)
encoded = layers.Dense(128, activation='relu')(flat)
encoded = layers.Dense(64, activation='relu')(encoded)
encoded = layers.Dense(encoding_dim, activation='relu')(encoded)

# Decoder path - reconstructs the flattened embedding space
decoded = layers.Dense(64, activation='relu')(encoded)
decoded = layers.Dense(128, activation='relu')(decoded)
decoded = layers.Dense(max_len * embedding_dim, activation='sigmoid')(decoded)

# Full model
autoencoder = Model(autoencoder_input, decoded, name='autoencoder')
autoencoder.compile(optimizer='adam', loss='mse')

print("\n=== Autoencoder Architecture ===")
autoencoder.summary()

print("\n=== Training Autoencoder ===")

# Create target: embedded representations (what we want to reconstruct)
print("Creating embedding targets...")
# Use embedding layer to get representations
embed_layer = layers.Embedding(input_dim=max_words, output_dim=embedding_dim)
# We'll embed in batches to avoid memory issues
batch_size_embed = 5000
X_train_embedded_list = []
for i in range(0, len(X_train), batch_size_embed):
    batch = X_train[i:i+batch_size_embed]
    embedded_batch = embed_layer(batch).numpy()  # [batch_size, max_len, embedding_dim]
    flattened = embedded_batch.reshape(embedded_batch.shape[0], -1).astype('float32')  # [batch_size, 1280]
    X_train_embedded_list.append(flattened)
X_train_embedded = np.vstack(X_train_embedded_list)

X_test_embedded_list = []
for i in range(0, len(X_test), batch_size_embed):
    batch = X_test[i:i+batch_size_embed]
    embedded_batch = embed_layer(batch).numpy()
    flattened = embedded_batch.reshape(embedded_batch.shape[0], -1).astype('float32')
    X_test_embedded_list.append(flattened)
X_test_embedded = np.vstack(X_test_embedded_list)

print(f"Created embedding targets: {X_train_embedded.shape}")

# Get normal samples embedded
normal_idx = np.where(y_train == 0)[0]
X_train_normal_embedded = X_train_embedded[normal_idx]
print(f"Training samples (real news embeddings only): {X_train_normal_embedded.shape[0]}")

# Train autoencoder to reconstruct embedding space
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = autoencoder.fit(
    X_train[normal_idx], X_train_normal_embedded,  # Input: tokens, Output: embeddings
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=1
)

print("\n✓ Training completed!")

Training samples (real news only): 599731

Building end-to-end autoencoder with embedding layer...

=== Autoencoder Architecture ===


Model: "autoencoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_13 (InputLayer)     │ (None, 20)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_5 (Embedding)         │ (None, 20, 64)         │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_9 (Flatten)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_24 (Dense)                │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_25 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_26 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_27 (Dense)                │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_28 (Dense)                │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_29 (Dense)                │ (None, 1280)           │       165,120 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,629,856 (6.22 MB)

Training samples (real news only): 599731

Building end-to-end autoencoder with embedding layer...

=== Autoencoder Architecture ===


...

 Trainable params: 1,629,856 (6.22 MB)

 Non-trainable params: 0 (0.00 B)


=== Training Autoencoder ===
Creating embedding targets...


MemoryError: Unable to allocate 24.4 MiB for an array with shape (5000, 20, 64) and data type float32

: 

In [ ]:
# Plot training history
plt.figure(figsize=(10, 5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.title('Autoencoder Training History')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(BASE_DIR / 'results' / 'figures' / 'autoencoder_training.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Final training loss: {history.history['loss'][-1]:.6f}")
print(f"Final validation loss: {history.history['val_loss'][-1]:.6f}")

## Calculate Reconstruction Errors

In [ ]:
# Calculate reconstruction errors for all samples using embeddings
print("Calculating reconstruction errors...")
X_train_pred = autoencoder.predict(X_train, verbose=0)
X_test_pred = autoencoder.predict(X_test, verbose=0)

# Compare against embedded targets (not raw tokens)
# MAE between true embeddings and reconstructed embeddings
train_mae = np.mean(np.abs(X_train_embedded - X_train_pred), axis=1)
test_mae = np.mean(np.abs(X_test_embedded - X_test_pred), axis=1)

print(f"Train reconstruction error (MAE) - Min: {train_mae.min():.6f}, Max: {train_mae.max():.6f}, Mean: {train_mae.mean():.6f}")
print(f"Test reconstruction error (MAE) - Min: {test_mae.min():.6f}, Max: {test_mae.max():.6f}, Mean: {test_mae.mean():.6f}")

## Define Anomaly Detection Threshold

In [ ]:
# Define threshold based on percentile of training errors
threshold_percentile = 95
threshold = np.percentile(train_mae, threshold_percentile)

print(f"\n=== Anomaly Detection Threshold ===")
print(f"Threshold (95th percentile of training errors): {threshold:.6f}")

# Visualize error distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Train errors
axes[0].hist(train_mae, bins=50, alpha=0.7, label='Train errors', edgecolor='black')
axes[0].axvline(threshold, color='red', linestyle='--', linewidth=2, label=f'Threshold: {threshold:.4f}')
axes[0].set_xlabel('Reconstruction Error (MAE)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Training Set - Reconstruction Error Distribution')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Test errors
axes[1].hist(test_mae, bins=50, alpha=0.7, label='Test errors', color='orange', edgecolor='black')
axes[1].axvline(threshold, color='red', linestyle='--', linewidth=2, label=f'Threshold: {threshold:.4f}')
axes[1].set_xlabel('Reconstruction Error (MAE)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Test Set - Reconstruction Error Distribution')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(BASE_DIR / 'results' / 'figures' / 'reconstruction_error_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

## Detect Anomalies

In [ ]:
# Classify samples as normal (0) or anomalous (1)
train_anomalies = (train_mae > threshold).astype(int)
test_anomalies = (test_mae > threshold).astype(int)

print(f"\n=== Anomaly Detection Results ===")
print(f"Training set:")
print(f"  - Normal samples: {(train_anomalies == 0).sum()}")
print(f"  - Anomalies detected: {(train_anomalies == 1).sum()}")
print(f"  - Anomaly rate: {(train_anomalies == 1).sum() / len(train_anomalies) * 100:.2f}%")

print(f"\nTest set:")
print(f"  - Normal samples: {(test_anomalies == 0).sum()}")
print(f"  - Anomalies detected: {(test_anomalies == 1).sum()}")
print(f"  - Anomaly rate: {(test_anomalies == 1).sum() / len(test_anomalies) * 100:.2f}%")

## Evaluate Anomaly Detection Results

In [ ]:
# Evaluate on test set (compare with actual labels)
# Note: y_test has real labels (0=real, 1=fake)
# test_anomalies has detector labels (0=normal, 1=anomaly)

from sklearn.metrics import confusion_matrix, classification_report, precision_score, recall_score, f1_score

print("=== Classification Metrics (Test Set) ===\n")

# Compare detected anomalies with actual fake news
print("Assuming: Fake news (label=1) should be detected as anomalies")
print("Ground truth: y_test (0=real, 1=fake)")
print("Detection: test_anomalies (0=normal, 1=anomaly)\n")

# Calculate metrics
accuracy = accuracy_score(y_test, test_anomalies)
precision = precision_score(y_test, test_anomalies, zero_division=0)
recall = recall_score(y_test, test_anomalies, zero_division=0)
f1 = f1_score(y_test, test_anomalies, zero_division=0)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")

# ROC-AUC using reconstruction errors as scores
roc_auc = roc_auc_score(y_test, test_mae)
print(f"\nROC-AUC Score (using reconstruction error): {roc_auc:.4f}")

# Confusion matrix
cm = confusion_matrix(y_test, test_anomalies)
print(f"\nConfusion Matrix:")
print(cm)

# Classification report
print(f"\nDetailed Classification Report:")
print(classification_report(y_test, test_anomalies, target_names=['Real News', 'Fake News'], zero_division=0))

In [ ]:
# Visualizations
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix Heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Real News', 'Fake News'],
            yticklabels=['Real News', 'Fake News'])
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')
axes[0].set_title('Confusion Matrix - Anomaly Detection')

# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, test_mae)
roc_auc_computed = auc(fpr, tpr)

axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc_computed:.4f})')
axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve - Anomaly Detection')
axes[1].legend(loc="lower right")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(BASE_DIR / 'results' / 'figures' / 'anomaly_detection_evaluation.png', dpi=300, bbox_inches='tight')
plt.show()

## Analyze Detected Anomalies

In [ ]:
# Get indices of detected anomalies and correctly identified fake news
anomaly_indices = np.where(test_anomalies == 1)[0]
correctly_detected_fake = anomaly_indices[y_test[anomaly_indices] == 1]
false_positive_indices = anomaly_indices[y_test[anomaly_indices] == 0]

print("=== Anomaly Analysis ===\n")
print(f"Total anomalies detected: {len(anomaly_indices)}")
print(f"True Positives (detected as anomaly, actually fake): {len(correctly_detected_fake)}")
print(f"False Positives (detected as anomaly, actually real): {len(false_positive_indices)}")

print(f"\n=== Reconstruction Error Statistics by True Label ===")
real_news_errors = test_mae[y_test == 0]
fake_news_errors = test_mae[y_test == 1]

print(f"\nReal News (Label=0):")
print(f"  - Count: {len(real_news_errors)}")
print(f"  - Mean error: {real_news_errors.mean():.6f}")
print(f"  - Std error: {real_news_errors.std():.6f}")
print(f"  - Min error: {real_news_errors.min():.6f}")
print(f"  - Max error: {real_news_errors.max():.6f}")

print(f"\nFake News (Label=1):")
print(f"  - Count: {len(fake_news_errors)}")
print(f"  - Mean error: {fake_news_errors.mean():.6f}")
print(f"  - Std error: {fake_news_errors.std():.6f}")
print(f"  - Min error: {fake_news_errors.min():.6f}")
print(f"  - Max error: {fake_news_errors.max():.6f}")

# Percentage of each class above threshold
real_above_threshold = (real_news_errors > threshold).sum() / len(real_news_errors) * 100
fake_above_threshold = (fake_news_errors > threshold).sum() / len(fake_news_errors) * 100

print(f"\n=== Percentage Above Threshold ({threshold:.6f}) ===")
print(f"Real News: {real_above_threshold:.2f}%")
print(f"Fake News: {fake_above_threshold:.2f}%")

In [ ]:
# Visualize reconstruction errors by label
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot
bp_data = [real_news_errors, fake_news_errors]
bp = axes[0].boxplot(bp_data, labels=['Real News', 'Fake News'], patch_artist=True)
for patch, color in zip(bp['boxes'], ['lightblue', 'lightcoral']):
    patch.set_facecolor(color)
axes[0].axhline(threshold, color='red', linestyle='--', linewidth=2, label=f'Threshold: {threshold:.4f}')
axes[0].set_ylabel('Reconstruction Error (MAE)')
axes[0].set_title('Reconstruction Error Distribution by Label')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# Stacked bar chart
categories = ['Real News', 'Fake News']
below_threshold = [
    (real_news_errors <= threshold).sum(),
    (fake_news_errors <= threshold).sum()
]
above_threshold = [
    (real_news_errors > threshold).sum(),
    (fake_news_errors > threshold).sum()
]

x = np.arange(len(categories))
width = 0.6

axes[1].bar(x, below_threshold, width, label='Below Threshold (Normal)', color='lightgreen')
axes[1].bar(x, above_threshold, width, bottom=below_threshold, label='Above Threshold (Anomaly)', color='lightcoral')
axes[1].set_ylabel('Number of Samples')
axes[1].set_title('Classification by Threshold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(categories)
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(BASE_DIR / 'results' / 'figures' / 'anomaly_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

## Save Model and Results

In [ ]:
# Save the autoencoder model
print("Saving models and results...")

# Save models
autoencoder.save(MODELS_DIR / 'autoencoder_embedding.h5')
encoder.save(MODELS_DIR / 'autoencoder_encoder_embedding.h5')
decoder.save(MODELS_DIR / 'autoencoder_decoder_embedding.h5')

# Save tokenizer
import pickle
with open(MODELS_DIR / 'tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

# Save threshold and metadata
import json
metadata = {
    'model_type': 'embedding_autoencoder',
    'threshold': float(threshold),
    'threshold_percentile': threshold_percentile,
    'max_len': int(max_len),
    'max_words': int(max_words),
    'embedding_dim': int(embedding_dim),
    'encoding_dim': int(encoding_dim),
    'train_sample_count': int(len(X_train_normal)),
    'accuracy': float(accuracy),
    'precision': float(precision),
    'recall': float(recall),
    'f1_score': float(f1),
    'roc_auc': float(roc_auc)
}

with open(MODELS_DIR / 'autoencoder_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

# Save reconstruction errors
dump(train_mae, MODELS_DIR / 'train_reconstruction_errors.joblib')
dump(test_mae, MODELS_DIR / 'test_reconstruction_errors.joblib')
dump(test_anomalies, MODELS_DIR / 'test_predictions.joblib')

print(f"\n✓ Models saved:")
print(f"  - Autoencoder: {MODELS_DIR / 'autoencoder_embedding.h5'}")
print(f"  - Encoder: {MODELS_DIR / 'autoencoder_encoder_embedding.h5'}")
print(f"  - Decoder: {MODELS_DIR / 'autoencoder_decoder_embedding.h5'}")
print(f"  - Tokenizer: {MODELS_DIR / 'tokenizer.pkl'}")
print(f"  - Metadata: {MODELS_DIR / 'autoencoder_metadata.json'}")
print(f"  - Errors: {MODELS_DIR / 'train_reconstruction_errors.joblib'}")
print(f"           {MODELS_DIR / 'test_reconstruction_errors.joblib'}")
print(f"           {MODELS_DIR / 'test_predictions.joblib'}")

## Summary

In [ ]:
print("="*60)
print("AUTOENCODER ANOMALY DETECTION - SUMMARY")
print("="*60)
print(f"\n📊 Model Architecture:")
print(f"   Input: Token sequences (length={max_len})")
print(f"   Embedding: {embedding_dim}-dimensional vectors")
print(f"   Bottleneck: {encoding_dim} dimensions")
print(f"   Vocabulary size: {max_words}")
print(f"   Architecture: Embedding → Flatten → Dense(128) → Dense(64) → Dense({encoding_dim}) → Dense(128) → Dense({max_len*embedding_dim}) → Reshape → Flatten")

print(f"\n📝 Data:")
print(f"   Training samples (real news): {X_train_normal.shape[0]}")
print(f"   Test samples: {X_test.shape[0]}")
print(f"   Fake news in test: {(y_test == 1).sum()}")

print(f"\n🎯 Anomaly Detection:")
print(f"   Threshold (95th percentile): {threshold:.6f}")
print(f"   Detection Method: Reconstruction Error (MAE)")

print(f"\n📈 Test Set Performance:")
print(f"   Accuracy:  {accuracy:.4f}")
print(f"   Precision: {precision:.4f}")
print(f"   Recall:    {recall:.4f}")
print(f"   F1-Score:  {f1:.4f}")
print(f"   ROC-AUC:   {roc_auc:.4f}")

print(f"\n✅ Results:")
print(f"   True Positives (Fake detected):  {len(correctly_detected_fake)}")
print(f"   False Positives (Real as Fake):  {len(false_positive_indices)}")
print(f"   Total Anomalies Detected:        {len(anomaly_indices)}")

print(f"\n💾 Output Files:")
print(f"   ✓ Models saved to: {MODELS_DIR}/")
print(f"   ✓ Figures saved to: {BASE_DIR / 'results' / 'figures'}/")
print("\n" + "="*60)